# Dataset

- Dataset API is mainly for Scala and Java
- In PySpark, we generally use DataFrame, not typed Dataset
- Use Scala to run this notebook

## Create a typed Dataset

In [0]:
%scala
case class Employee(id: Int, name: String, dept: String, salary: Double)

import spark.implicits._

val employees = Seq(
  Employee(1, "Alice", "IT", 90000),
  Employee(2, "Bob", "HR", 60000),
  Employee(3, "Cathy", "IT", 95000),
  Employee(4, "David", "Finance", 70000),
  Employee(5, "Eva", "HR", 65000)
)

val empDS = employees.toDS()

empDS.show()

## Filter records

In [0]:
%scala
val highSalaryDS = empDS.filter(emp => emp.salary > 70000)

highSalaryDS.show()

## Map records

In [0]:
%scala
val employeeNamesDS = empDS.map(emp => emp.name)

employeeNamesDS.show()

## Select specific columns

In [0]:
%scala
val selectedDF = empDS.select("name", "dept", "salary")

selectedDF.show()

## Group and aggregate

In [0]:
%scala
import org.apache.spark.sql.functions._

val deptSummary = empDS
  .groupBy("dept")
  .agg(
    avg("salary").alias("avg_salary"),
    max("salary").alias("max_salary"),
    count("id").alias("employee_count")
  )

deptSummary.show()

## Join two Datasets

In [0]:
%scala
case class Department(dept: String, location: String)

val departments = Seq(
  Department("IT", "Bangalore"),
  Department("HR", "Chennai"),
  Department("Finance", "Mumbai")
)

val deptDS = departments.toDS()

val joinedDS = empDS.joinWith(deptDS, empDS("dept") === deptDS("dept"))

joinedDS.show(false)

## Convert Dataset to DataFrame

In [0]:
%scala
val empDF = empDS.toDF()

empDF.show()
empDF.printSchema()

## Full End-to-End Dataset Demo

In [0]:
%scala
case class Employee(id: Int, name: String, dept: String, salary: Double)
case class Department(dept: String, location: String)

import spark.implicits._
import org.apache.spark.sql.functions._

val employees = Seq(
  Employee(1, "Alice", "IT", 90000),
  Employee(2, "Bob", "HR", 60000),
  Employee(3, "Cathy", "IT", 95000),
  Employee(4, "David", "Finance", 70000),
  Employee(5, "Eva", "HR", 65000)
)

val departments = Seq(
  Department("IT", "Bangalore"),
  Department("HR", "Chennai"),
  Department("Finance", "Mumbai")
)

val empDS = employees.toDS()
val deptDS = departments.toDS()

println("Original Dataset")
empDS.show()

println("Filter: salary > 70000")
empDS.filter(_.salary > 70000).show()

println("Map: extract names")
empDS.map(_.name).show()

println("Group by department")
empDS.groupBy("dept")
  .agg(
    avg("salary").alias("avg_salary"),
    count("id").alias("employee_count")
  )
  .show()

println("Typed groupByKey")
empDS.groupByKey(_.dept).count().show()

println("Join with department dataset")
empDS.joinWith(deptDS, empDS("dept") === deptDS("dept")).show(false)

println("Convert to DataFrame")
val empDF = empDS.toDF()
empDF.show()

println("Convert to RDD")
val empRDD = empDS.rdd
println(empRDD.collect().mkString(", "))